In [27]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import torch
from torch.utils.data import DataLoader
from scripts.dataset import PortfolioDataset,multitask_collate_fn

test_dataset = PortfolioDataset(
    "../data/cleaned/processed/test_processed.jsonl"
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=multitask_collate_fn
)

In [28]:
import torch
import torch.nn as nn
from transformers import AutoModel

class MultitaskXLM(nn.Module):
    def __init__(
        self,
        model_name="xlm-roberta-base",
        num_domain_labels=10,
        num_ner_labels=3,
    ):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.domain_dropout = nn.Dropout(0.2)
        self.ner_dropout = nn.Dropout(0.1)
        self.domain_classifier = nn.Linear(hidden_size, num_domain_labels)
        self.ner_classifier = nn.Linear(hidden_size, num_ner_labels)

    def forward(self, input_ids, attention_mask, domain_labels=None, ner_labels=None):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        hidden_states = outputs.last_hidden_state.float()

        # cls_vector = hidden_states[:, 0, :]
        mask_expanded = attention_mask.unsqueeze(-1).float()
        mean_pooled = (hidden_states * mask_expanded).sum(1) / mask_expanded.sum(1)

        domain_logits = self.domain_classifier(self.domain_dropout(mean_pooled)) # instead of cls for better representation of the sequence
        ner_logits = self.ner_classifier(self.ner_dropout(hidden_states))

        loss = None

        if domain_labels is not None and ner_labels is not None:
            domain_loss_fn = nn.BCEWithLogitsLoss()
            ner_loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

            domain_loss = domain_loss_fn(domain_logits, domain_labels.float())

            ner_loss = ner_loss_fn(
                ner_logits.reshape(-1, ner_logits.shape[-1]),
                ner_labels.reshape(-1)
            )

            loss = domain_loss * 0.7 + ner_loss * 0.3 # Weighted loss because NER has way higher results that Domain classification

        return {
            "domain_logits": domain_logits,
            "ner_logits": ner_logits,
            "loss": loss
        }
    
model = MultitaskXLM(num_domain_labels=10, num_ner_labels=3)
model.load_state_dict(torch.load("../models/best_model.pt", map_location=torch.device('cpu')))
model.eval()

MultitaskXLM(
  (encoder): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
           

In [29]:
model.load_state_dict(torch.load("../models/best_model.pt"))
model.eval()

MultitaskXLM(
  (encoder): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
           

In [30]:
from sklearn.metrics import f1_score
import torch

def eval_step(model, dataloader, device):
    model.eval()

    total_loss = 0

    all_domain_preds = []
    all_domain_labels = []

    all_ner_preds = []
    all_ner_labels = []

    with torch.inference_mode():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            domain_labels = batch["domain_labels"].to(device)
            ner_labels = batch["ner_labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                domain_labels=domain_labels,
                ner_labels=ner_labels,
            )

            loss = outputs["loss"]
            total_loss += loss.item()

            # Domain predictions
            domain_logits = outputs["domain_logits"]
            domain_preds = (torch.sigmoid(domain_logits) > 0.5).int()

            all_domain_preds.append(domain_preds.cpu())
            all_domain_labels.append(domain_labels.cpu().int())

            # NER predictions
            ner_logits = outputs["ner_logits"]
            ner_preds = ner_logits.argmax(dim=-1)

            mask = ner_labels != -100

            all_ner_preds.append(ner_preds[mask].cpu())
            all_ner_labels.append(ner_labels[mask].cpu())

    avg_loss = total_loss / len(dataloader)

    all_domain_preds = torch.cat(all_domain_preds).numpy()
    all_domain_labels = torch.cat(all_domain_labels).numpy()

    all_ner_preds = torch.cat(all_ner_preds).numpy()
    all_ner_labels = torch.cat(all_ner_labels).numpy()

    domain_micro_f1 = f1_score(
        all_domain_labels,
        all_domain_preds,
        average="micro",
        zero_division=0
    )

    domain_macro_f1 = f1_score(
        all_domain_labels,
        all_domain_preds,
        average="macro",
        zero_division=0
    )

    ner_micro_f1 = f1_score(
        all_ner_labels,
        all_ner_preds,
        average="micro",
        zero_division=0
    )

    ner_macro_f1 = f1_score(
        all_ner_labels,
        all_ner_preds,
        average="macro",
        zero_division=0
    )

    return {
        "loss": avg_loss,
        "domain_micro_f1": domain_micro_f1,
        "domain_macro_f1": domain_macro_f1,
        "ner_micro_f1": ner_micro_f1,
        "ner_macro_f1": ner_macro_f1,
    }

In [31]:
test_metrics = eval_step(model, test_dataloader, "cpu")
print(test_metrics)

{'loss': 0.0893889603515466, 'domain_micro_f1': 0.8912655971479501, 'domain_macro_f1': 0.8981459575345919, 'ner_micro_f1': 0.9740019207024283, 'ner_macro_f1': 0.9460522363970583}


In [33]:
from transformers import AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
model.eval()

tex2 = "The Smart Retail Management System (SRMS) is a cloud-based enterprise application developed for medium-sized retail businesses to automate inventory management, sales tracking, customer relationship management, and business analytics."
text3 = "Developed a multilingual AI system based on Transformer architectures to automatically analyze and classify technical portfolio content. The project combines Named Entity Recognition (NER) and multi-label text classification in a single Multi-Task Learning (MTL) model using XLM-RoBERTa."
text = "Developed a scalable e-commerce platform using React, Next.js, and TypeScript for the frontend with a Node.js and Express backend. Integrated Redis caching and PostgreSQL for performance optimization. Containerized the application with Docker and deployed it on Kubernetes using Helm charts and GitHub Actions CI/CD pipelines. Implemented OAuth2 authentication, JWT session management, and Web Application Firewall protection with Cloudflare."
text1 = "Designed a smart irrigation system powered by ESP32 microcontrollers and MQTT communication. Sensor data for soil humidity and temperature was transmitted to AWS IoT Core and visualized through a Flutter mobile application. Implemented Bluetooth Low Energy pairing and offline synchronization for remote agricultural environments with unstable connectivity."
text11= "Built a real-time anomaly detection pipeline using Python, Kafka, and Spark Structured Streaming, deployed on Kubernetes with Helm charts, monitored via Prometheus and Grafana, models trained with scikit-learn and tracked in MLflow, infrastructure provisioned by Terraform on AWS, with a React dashboard consuming a FastAPI backend secured by OAuth2."

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.inference_mode():
    outputs = model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"]
    )

domain_probs = torch.sigmoid(outputs["domain_logits"])[0]
domain_preds = (domain_probs > 0.1).int()

print(f"TEXT : {text} \n")
print(domain_probs)
print(domain_preds)

domain_names = [
    "Web Frontend",
    "Web Backend",
    "Mobile Development",
    "DevOps and Cloud Infrastructure",
    "Data Engineering",
    "Machine Learning and AI",
    "Cybersecurity",
    "Embedded Systems and IoT",
    "High Performance and Quantum Computing",
    "Other",
]

for i, pred in enumerate(domain_preds):
    if pred == 1:
        print(f"\n Domains predicted : {domain_names[i], float(domain_probs[i])}")

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

ner_preds = outputs["ner_logits"].argmax(dim=-1)[0].cpu().tolist()

id2label = {
    0: "O",
    1: "B-TECH",
    2: "I-TECH"
}

print (f"\n Technologies :")
for token, pred in zip(tokens, ner_preds):
    if pred != 0:
        print(token, id2label[pred])

TEXT : Developed a scalable e-commerce platform using React, Next.js, and TypeScript for the frontend with a Node.js and Express backend. Integrated Redis caching and PostgreSQL for performance optimization. Containerized the application with Docker and deployed it on Kubernetes using Helm charts and GitHub Actions CI/CD pipelines. Implemented OAuth2 authentication, JWT session management, and Web Application Firewall protection with Cloudflare. 

tensor([0.9959, 0.3090, 0.0088, 0.1553, 0.0116, 0.0078, 0.0987, 0.0051, 0.0082,
        0.0069], device='cuda:0')
tensor([1, 1, 0, 1, 0, 0, 0, 0, 0, 0], device='cuda:0', dtype=torch.int32)

 Domains predicted : ('Web Frontend', 0.9959055185317993)

 Domains predicted : ('Web Backend', 0.30901655554771423)

 Domains predicted : ('DevOps and Cloud Infrastructure', 0.15525615215301514)

 Technologies :
▁Re B-TECH
act I-TECH
▁Next B-TECH
. I-TECH
js I-TECH
▁Type B-TECH
Script I-TECH
▁No B-TECH
de I-TECH
. I-TECH
js I-TECH
▁Express B-TECH
▁Red B-T

In [22]:
import copy
import torch

quantized_model = torch.quantization.quantize_dynamic(
    copy.deepcopy(model),
    {torch.nn.Linear},  # all linear layers this time
    dtype=torch.qint8
)

In [23]:
test_metrics = eval_step(quantized_model, test_dataloader, "cpu")
print(test_metrics)

{'loss': 0.11510618329048157, 'domain_micro_f1': 0.8586956521739131, 'domain_macro_f1': 0.8638375412234465, 'ner_micro_f1': 0.9511592811085197, 'ner_macro_f1': 0.8931387208009715}


In [40]:
import copy
import torch

optimized_model = copy.deepcopy(model)
optimized_model.to("cpu")
optimized_model.eval()

optimized_model = torch.quantization.quantize_dynamic(
    optimized_model,
    {torch.nn.Linear},
    dtype=torch.qint8
)

In [41]:
test_metrics = eval_step(optimized_model, test_dataloader, "cpu")
print(test_metrics)

{'loss': 0.11510618329048157, 'domain_micro_f1': 0.8586956521739131, 'domain_macro_f1': 0.8638375412234465, 'ner_micro_f1': 0.9511592811085197, 'ner_macro_f1': 0.8931387208009715}


In [44]:
from transformers import AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
device = torch.device("cpu")

optimized_model.to(device)
optimized_model.eval()

tex2 = "The Smart Retail Management System (SRMS) is a cloud-based enterprise application developed for medium-sized retail businesses to automate inventory management, sales tracking, customer relationship management, and business analytics."
text3 = "Developed a multilingual AI system based on Transformer architectures to automatically analyze and classify technical portfolio content. The project combines Named Entity Recognition (NER) and multi-label text classification in a single Multi-Task Learning (MTL) model using XLM-RoBERTa."
text = "Developed a scalable e-commerce platform using React, Next.js, and TypeScript for the frontend with a Node.js and Express backend. Integrated Redis caching and PostgreSQL for performance optimization. Containerized the application with Docker and deployed it on Kubernetes using Helm charts and GitHub Actions CI/CD pipelines. Implemented OAuth2 authentication, JWT session management, and Web Application Firewall protection with Cloudflare."
text1 = "Designed a smart irrigation system powered by ESP32 microcontrollers and MQTT communication. Sensor data for soil humidity and temperature was transmitted to AWS IoT Core and visualized through a Flutter mobile application. Implemented Bluetooth Low Energy pairing and offline synchronization for remote agricultural environments with unstable connectivity."
text11= "Built a real-time anomaly detection pipeline using Python, Kafka, and Spark Structured Streaming, deployed on Kubernetes with Helm charts, monitored via Prometheus and Grafana, models trained with scikit-learn and tracked in MLflow, infrastructure provisioned by Terraform on AWS, with a React dashboard consuming a FastAPI backend secured by OAuth2."

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.inference_mode():
    outputs = optimized_model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"]
    )

domain_probs = torch.sigmoid(outputs["domain_logits"])[0]
domain_preds = (domain_probs > 0.1).int()

print(f"TEXT : {text} \n")
print(domain_probs)
print(domain_preds)

domain_names = [
    "Web Frontend",
    "Web Backend",
    "Mobile Development",
    "DevOps and Cloud Infrastructure",
    "Data Engineering",
    "Machine Learning and AI",
    "Cybersecurity",
    "Embedded Systems and IoT",
    "High Performance and Quantum Computing",
    "Other",
]

for i, pred in enumerate(domain_preds):
    if pred == 1:
        print(f"\n Domains predicted : {domain_names[i], float(domain_probs[i])}")

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

ner_preds = outputs["ner_logits"].argmax(dim=-1)[0].cpu().tolist()

id2label = {
    0: "O",
    1: "B-TECH",
    2: "I-TECH"
}

print (f"\n Technologies :")
for token, pred in zip(tokens, ner_preds):
    if pred != 0:
        print(token, id2label[pred])

TEXT : Developed a scalable e-commerce platform using React, Next.js, and TypeScript for the frontend with a Node.js and Express backend. Integrated Redis caching and PostgreSQL for performance optimization. Containerized the application with Docker and deployed it on Kubernetes using Helm charts and GitHub Actions CI/CD pipelines. Implemented OAuth2 authentication, JWT session management, and Web Application Firewall protection with Cloudflare. 

tensor([0.9935, 0.2847, 0.0328, 0.5967, 0.0286, 0.0159, 0.2091, 0.0104, 0.0287,
        0.0145])
tensor([1, 1, 0, 1, 0, 0, 1, 0, 0, 0], dtype=torch.int32)

 Domains predicted : ('Web Frontend', 0.9934939742088318)

 Domains predicted : ('Web Backend', 0.2846662104129791)

 Domains predicted : ('DevOps and Cloud Infrastructure', 0.5966899991035461)

 Domains predicted : ('Cybersecurity', 0.2090863734483719)

 Technologies :
▁Re B-TECH
act I-TECH
▁Next B-TECH
. I-TECH
js I-TECH
▁Type B-TECH
Script I-TECH
▁No B-TECH
de I-TECH
. I-TECH
js I-TECH


In [ ]:
torch.save(optimized_model.state_dict(), "../models/model_optimized.pt")